# Auto BC Processing

- gathers huc to huc dictionary data and creates sequence

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
#imports
import os
import pathlib as pl

## Set directories and user defined variables

In [ ]:
#set working directory and folder variables
os.chdir('..')
os.chdir('..')

In [4]:
home = pl.Path(os.getcwd())
print('home is at ',home)
from src.sequence import *

inputs = home/'inputs'
outputs_base = home/'outputs'

home is at  C:\git\hdf_reader


In [5]:
#user to set variables for the project
project = 'wy_fy22'

In [6]:
#get upstream models
with open(inputs/project/'dictionaries'/'HUC10_outflow_toHUC10.json') as src:
    huc_connect_huc = json.load(src)

In [7]:
hucs_df = pd.DataFrame.from_dict(huc_connect_huc,orient='index',columns=['ds_huc'])
hucs_df['huc'] = hucs_df.index

In [8]:
#create lists for analysis
to_hucs = hucs_df.ds_huc.to_list()
from_hucs = hucs_df.huc.to_list()

In [9]:
#identify downstream count and flow path (aka sequence)
hucs_df['ds_count'] = hucs_df.apply(lambda x: ds_count(hucs_df,'huc','ds_huc',x.huc,x.ds_huc,to_hucs,from_hucs)[0],axis = 1)
hucs_df['ds_sequence'] = hucs_df.apply(lambda x: ds_count(hucs_df,'huc','ds_huc',x.huc,x.ds_huc,to_hucs,from_hucs)[1],axis = 1)

In [10]:
#identify upstream count and flow path (aka sequence)
hucs_df['us_count'] = hucs_df.apply(lambda x: us_count(hucs_df,'huc','ds_huc',x.huc,x.ds_huc,to_hucs,from_hucs)[0],axis = 1)
hucs_df['us_sequence'] = hucs_df.apply(lambda x: us_count(hucs_df,'huc','ds_huc',x.huc,x.ds_huc,to_hucs,from_hucs)[1],axis = 1)

In [11]:
hucs_df

,ds_huc,huc,ds_count,ds_sequence,us_count,us_sequence
1404010102,1404010107,1404010102,10,"[1404010107, 1404010109, 1404010111, 140401011...",0,[]
1404010103,1404010107,1404010103,10,"[1404010107, 1404010109, 1404010111, 140401011...",0,[]
1404010104,1404010107,1404010104,10,"[1404010107, 1404010109, 1404010111, 140401011...",0,[]
1404010105,1404010107,1404010105,10,"[1404010107, 1404010109, 1404010111, 140401011...",0,[]
1404010106,1404010109,1404010106,9,"[1404010109, 1404010111, 1404010113, 140401030...",0,[]
...,...,...,...,...,...,...
1404010903,1404010610,1404010903,1,[1404010610],0,[]
1404020004,1404020005,1404020004,1,[1404020005],0,[]
1404020005,N/A,1404020005,0,[],1,[1404020004]
1404020007,1404020008,1404020007,1,[1404020008],0,[]


In [12]:
hucs_df.set_index('huc',inplace=True)

In [15]:
#create FY field and set as null
hucs_df['sequence'] = None
start = 1
end = hucs_df.us_count.max() + start
sequencer_dict = {}
for cycle in np.arange(start,end+1,1):
    print(cycle)
    cycles_left = end - cycle
    ready_hucs = rule(hucs_df,cycle,cycles_left)
    sequencer_dict[cycle] = ready_hucs
    print('\n')

1
['1404010610', '1404020005', '1404020008']


2
['1404010609', '1404010903', '1404020004', '1404020007']


3
['1404010607', '1404010608']


4
['1404010603', '1404010604', '1404010605']


5
['1404010601', '1404010602', '1404010710']


6
['1404010306', '1404010509', '1404010706', '1404010707', '1404010708', '1404010709']


7
['1404010113', '1404010301', '1404010303', '1404010304', '1404010305', '1404010406', '1404010503', '1404010504', '1404010506', '1404010507', '1404010508', '1404010702', '1404010703', '1404010704', '1404010705', '1404010803']


8
['1404010111', '1404010112', '1404010302', '1404010401', '1404010403', '1404010404', '1404010502', '1404010701', '1404010802']


9
['1404010109', '1404010110', '1404010402', '1404010801']


10
['1404010106', '1404010107', '1404010108']


11
['1404010102', '1404010103', '1404010104', '1404010105', '1404010206']


12
['1404010201', '1404010202', '1404010203', '1404010204', '1404010205']




In [21]:
import json

# Serialize data into file:
json.dump(str(sequencer_dict), open(inputs/project/'dictionaries'/"sequencer.json", 'w' ) )